In [ ]:
# pip install trimesh numpy scipy rtree pandas
# pip install manifold3d

import os
import itertools
import warnings
from pathlib import Path
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import pandas as pd
import trimesh
from trimesh.registration import icp

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, ScalarFormatter
from mpl_toolkits.mplot3d.art3d import Poly3DCollection


warnings.filterwarnings("ignore")

COLOR_CAD = "#1f77b4"
COLOR_RECON = "#ff7f0e"

EDGE_COLOR = "#2f2f2f"
EDGE_WIDTH = 0.10
EDGE_ALPHA = 0.45

FINAL_PNG = "final.png"
METRICS_CSV = "volumetric_error_report.csv"

PAIR_SPECS = [
    {
        "name": "Bearing",
        "cad": "input/Bearings_stls/Bearing_ref.stl",
        "recon": "input/Bearings_stls/Bearing_pred.stl",
        "cad_filename": "Bearing_ref.stl",
        "recon_filename": "Bearing_pred.stl",
        "folder_hint": "Bearings_stls",
        "first_row_seed": 101,
    },
    {
        "name": "Mount",
        "cad": "input/Mounts_stls/Mount_ref.stl",
        "recon": "input/Mounts_stls/Mount_pred.stl",
        "cad_filename": "Mount_ref.stl",
        "recon_filename": "Mount_pred.stl",
        "folder_hint": "Mounts_stls",
        "first_row_seed": 202,
    },
    {
        "name": "Plug",
        "cad": "input/plugs_stl/Plug_ref.stl",
        "recon": "input/plugs_stl/plug_pred.stl",
        "cad_filename": "Plug_ref.stl",
        "recon_filename": "plug_pred.stl",
        "folder_hint": "plugs_stl",
        "first_row_seed": 303,
    },
]

VIZ_ELEV = 20
VIZ_AZIM = -60
VIZ_MAX_FACES = 35000
INPUT_ROW_GAP_RANGE_MM = (22.0, 38.0)
INPUT_ROW_MIN_HALF_RANGE_MM = 35.0

def find_file_case_insensitive(filename, folder_hint=None, roots=("input",)):
    filename_lower = filename.lower()
    matches = []

    for root in roots:
        root = Path(root)

        if not root.exists():
            continue

        for p in root.rglob("*"):
            if p.is_file() and p.name.lower() == filename_lower:
                matches.append(p)

    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under {roots}")

    if folder_hint is not None:
        hint_lower = folder_hint.lower()
        hinted = [p for p in matches if hint_lower in str(p).lower()]

        if hinted:
            return str(hinted[0])

    return str(matches[0])

def resolve_path(path_candidate, filename, folder_hint=None):
    if path_candidate and os.path.exists(path_candidate):
        return path_candidate

    return find_file_case_insensitive(
        filename=filename,
        folder_hint=folder_hint,
    )

def load_mesh(path):
    mesh = trimesh.load(path, force="mesh")

    if not isinstance(mesh, trimesh.Trimesh):
        raise ValueError(f"Could not load a triangle mesh from {path}")

    mesh.remove_unreferenced_vertices()
    mesh.merge_vertices()

    try:
        trimesh.repair.fix_normals(mesh)
    except Exception:
        pass

    try:
        if mesh.volume < 0:
            mesh.invert()
    except Exception:
        pass

    return mesh

def projected_half_extent(mesh, direction):
    center = mesh.centroid
    values = (mesh.vertices - center) @ direction
    return float(np.max(np.abs(values)))

def randomized_input_copies_for_display(
    cad,
    recon,
    seed=0,
    gap_range_mm=(22.0, 38.0),
):
    rng = np.random.default_rng(seed)

    cad_plot = cad.copy()
    recon_plot = recon.copy()

    cad_center = cad_plot.centroid.copy()
    recon_center = recon_plot.centroid.copy()

    direction = rng.normal(size=3)
    direction[2] *= 0.35
    direction = direction / np.linalg.norm(direction)

    cad_half = projected_half_extent(cad_plot, direction)
    recon_half = projected_half_extent(recon_plot, direction)

    gap_mm = float(rng.uniform(gap_range_mm[0], gap_range_mm[1]))
    center_distance = cad_half + recon_half + gap_mm

    sign = rng.choice([-1.0, 1.0])

    global_offset = rng.uniform(-4.0, 4.0, size=3)
    global_offset[2] *= 0.5

    cad_target = global_offset - sign * direction * center_distance / 2.0
    recon_target = global_offset + sign * direction * center_distance / 2.0

    cad_plot.apply_translation(cad_target - cad_center)
    recon_plot.apply_translation(recon_target - recon_center)

    return cad_plot, recon_plot

def volume_metrics(cad, recon):
    cad_volume = abs(float(cad.volume))
    recon_volume = abs(float(recon.volume))

    if cad_volume <= 0:
        relative_error = np.nan
        accuracy = np.nan
    else:
        relative_error = abs(cad_volume - recon_volume) / cad_volume
        accuracy = 1.0 - relative_error

    return cad_volume, recon_volume, relative_error, accuracy

def pca_axes(mesh, n_sample=1000):
    n_sample = min(n_sample, max(len(mesh.faces), 100))

    pts = mesh.sample(n_sample)
    pts = pts - pts.mean(axis=0)

    _, _, vt = np.linalg.svd(pts, full_matrices=False)

    if np.linalg.det(vt) < 0:
        vt[2] *= -1

    return vt

def initial_rotations(recon, cad):
    rotations = [np.eye(3)]

    recon_axes = pca_axes(recon)
    cad_axes = pca_axes(cad)

    for sx, sy in itertools.product([1, -1], repeat=2):
        flip = np.diag([sx, sy, sx * sy])
        rotations.append(cad_axes.T @ flip @ recon_axes)

    for axis in [(1, 0, 0), (0, 1, 0), (0, 0, 1)]:
        rotations.append(
            trimesh.transformations.rotation_matrix(np.pi, axis)[:3, :3]
        )

    return rotations

def register(recon, cad, n_samples=1000, icp_iters=40):
    t_cad = np.eye(4)
    t_cad[:3, 3] = -cad.centroid

    if abs(recon.volume) > 0 and abs(cad.volume) > 0:
        s0 = (abs(cad.volume) / abs(recon.volume)) ** (1.0 / 3.0)
    else:
        s0 = 1.0

    coarse = np.eye(4)
    coarse[:3, :3] *= s0
    coarse[:3, 3] = -s0 * recon.centroid

    cad_centered = cad.copy()
    cad_centered.apply_transform(t_cad)

    src_points = trimesh.transform_points(
        recon.sample(n_samples),
        coarse,
    )

    best_cost = np.inf
    best_matrix = np.eye(4)

    for rotation in initial_rotations(recon, cad):
        init = np.eye(4)
        init[:3, :3] = rotation

        try:
            matrix, _, cost = icp(
                src_points,
                cad_centered,
                initial=init,
                scale=True,
                max_iterations=icp_iters,
            )
        except Exception:
            continue

        if cost < best_cost:
            best_cost = cost
            best_matrix = matrix

    inv_t_cad = np.linalg.inv(t_cad)

    final_transform = inv_t_cad @ best_matrix @ coarse
    coarse_transform = inv_t_cad @ coarse

    return final_transform, coarse_transform, best_cost

def add_mesh(
    ax,
    mesh,
    color,
    alpha=0.6,
    max_faces=35000,
    seed=0,
    show_edges=True,
):
    faces = mesh.faces

    if len(faces) > max_faces:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(faces), max_faces, replace=False)
        faces = faces[idx]

    triangles = mesh.vertices[faces]

    if show_edges:
        edge_rgba = matplotlib.colors.to_rgba(EDGE_COLOR, EDGE_ALPHA)
        edge_color = edge_rgba
        line_width = EDGE_WIDTH
    else:
        edge_color = "none"
        line_width = 0.0

    collection = Poly3DCollection(
        triangles,
        facecolor=color,
        edgecolor=edge_color,
        linewidths=line_width,
        alpha=alpha,
        antialiased=True,
    )

    ax.add_collection3d(collection)

def set_equal_bounds(
    ax,
    cad,
    recon,
    pad=0.06,
    fixed_center=None,
    fixed_half_range=None,
):
    lo = np.minimum(cad.bounds[0], recon.bounds[0])
    hi = np.maximum(cad.bounds[1], recon.bounds[1])

    if fixed_center is None:
        center = (lo + hi) / 2.0
        max_span = float((hi - lo).max())

        if fixed_half_range is None:
            half_range = max_span / 2.0 * (1.0 + pad)
        else:
            half_range = max(float(fixed_half_range), max_span / 2.0 * 1.03)

    else:
        center = np.asarray(fixed_center, dtype=float)

        max_extent = np.max(
            np.abs(
                np.vstack(
                    [
                        lo - center,
                        hi - center,
                    ]
                )
            )
        )

        if fixed_half_range is None:
            half_range = max_extent * (1.0 + pad)
        else:
            half_range = max(float(fixed_half_range), max_extent * 1.08)

    ax.set_xlim(center[0] - half_range, center[0] + half_range)
    ax.set_ylim(center[1] - half_range, center[1] + half_range)
    ax.set_zlim(center[2] - half_range, center[2] + half_range)

    try:
        ax.set_box_aspect((1, 1, 1))
    except Exception:
        pass

def style_axis(ax):
    ax.set_xlabel("x (mm)", fontsize=18, labelpad=8)
    ax.set_ylabel("y (mm)", fontsize=18, labelpad=8)
    ax.set_zlabel("z (mm)", fontsize=18, labelpad=8)

    formatter = ScalarFormatter(useOffset=False)
    formatter.set_scientific(False)

    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis.set_major_locator(MaxNLocator(nbins=3))
        axis.set_major_formatter(formatter)

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=16,
        pad=2,
        length=3.2,
        width=0.8,
    )

    try:
        ax.zaxis.set_tick_params(
            labelsize=16,
            pad=2,
            length=3.2,
            width=0.8,
        )
    except Exception:
        pass

    ax.grid(True, linewidth=0.38, alpha=0.30)

    try:
        ax.xaxis.pane.set_facecolor((1, 1, 1, 0.0))
        ax.yaxis.pane.set_facecolor((1, 1, 1, 0.0))
        ax.zaxis.pane.set_facecolor((1, 1, 1, 0.0))

        ax.xaxis.pane.set_edgecolor((1, 1, 1, 0.0))
        ax.yaxis.pane.set_edgecolor((1, 1, 1, 0.0))
        ax.zaxis.pane.set_edgecolor((1, 1, 1, 0.0))
    except Exception:
        pass

    try:
        ax.set_proj_type("ortho")
    except Exception:
        pass

def render_overlay_grid(results, out_path):
    n_rows = 3
    n_cols = len(results)

    fig = plt.figure(
        figsize=(6.9 * n_cols, 6.1 * n_rows),
        facecolor="white",
    )

    gs = fig.add_gridspec(
        n_rows,
        n_cols,
        wspace=0.22,
        hspace=0.16,
    )

    for col_idx, result in enumerate(results):
        for row_idx, step in enumerate(result["steps"]):
            cad = step["cad"]
            recon = step["recon"]

            ax = fig.add_subplot(
                gs[row_idx, col_idx],
                projection="3d",
            )

            add_mesh(
                ax,
                cad,
                COLOR_CAD,
                alpha=0.42,
                max_faces=VIZ_MAX_FACES,
                seed=1000 + 10 * row_idx + col_idx,
                show_edges=True,
            )

            add_mesh(
                ax,
                recon,
                COLOR_RECON,
                alpha=0.62,
                max_faces=VIZ_MAX_FACES,
                seed=2000 + 10 * row_idx + col_idx,
                show_edges=True,
            )

            if row_idx == 0:
                set_equal_bounds(
                    ax,
                    cad,
                    recon,
                    fixed_center=(0.0, 0.0, 0.0),
                    fixed_half_range=INPUT_ROW_MIN_HALF_RANGE_MM,
                )
            else:
                set_equal_bounds(
                    ax,
                    cad,
                    recon,
                    pad=0.08,
                )

            style_axis(ax)

            ax.view_init(
                elev=VIZ_ELEV,
                azim=VIZ_AZIM,
            )

    fig.savefig(
        out_path,
        dpi=240,
        facecolor="white",
    )

    plt.close(fig)

def analyze_pair(spec, n_samples=1000, icp_iters=40):
    cad_path = resolve_path(
        spec["cad"],
        spec["cad_filename"],
        spec["folder_hint"],
    )

    recon_path = resolve_path(
        spec["recon"],
        spec["recon_filename"],
        spec["folder_hint"],
    )

    cad = load_mesh(cad_path)
    recon = load_mesh(recon_path)

    cad_input_plot, recon_input_plot = randomized_input_copies_for_display(
        cad,
        recon,
        seed=spec.get("first_row_seed", 0),
        gap_range_mm=INPUT_ROW_GAP_RANGE_MM,
    )

    final_transform, coarse_transform, icp_cost = register(
        recon,
        cad,
        n_samples=n_samples,
        icp_iters=icp_iters,
    )

    recon_coarse = recon.copy()
    recon_coarse.apply_transform(coarse_transform)

    recon_icp = recon.copy()
    recon_icp.apply_transform(final_transform)

    cad_volume, recon_volume, rel_error, accuracy = volume_metrics(cad, recon_icp)

    metrics = {
        "part": spec["name"],
        "cad_volume_mm3": cad_volume,
        "reconstruction_volume_mm3": recon_volume,
        "relative_volume_error": rel_error,
        "relative_volume_error_percent": rel_error * 100.0,
        "volumetric_accuracy": accuracy,
        "icp_cost": icp_cost,
    }

    return {
        "name": spec["name"],
        "icp_cost": icp_cost,
        "metrics": metrics,
        "steps": [
            {
                "step": "input",
                "cad": cad_input_plot,
                "recon": recon_input_plot,
            },
            {
                "step": "coarse",
                "cad": cad.copy(),
                "recon": recon_coarse,
            },
            {
                "step": "icp",
                "cad": cad.copy(),
                "recon": recon_icp,
            },
        ],
    }

with open(os.devnull, "w") as devnull:
    with redirect_stdout(devnull), redirect_stderr(devnull):
        all_results = [
            analyze_pair(
                spec,
                n_samples=1000,
                icp_iters=40,
            )
            for spec in PAIR_SPECS
        ]

        render_overlay_grid(
            all_results,
            FINAL_PNG,
        )

metrics_df = pd.DataFrame([r["metrics"] for r in all_results])
metrics_df.to_csv(METRICS_CSV, index=False)

print(
    metrics_df.round(
        {
            "cad_volume_mm3": 3,
            "reconstruction_volume_mm3": 3,
            "relative_volume_error": 6,
            "relative_volume_error_percent": 3,
            "volumetric_accuracy": 6,
            "icp_cost": 6,
        }
    ).to_string(index=False)
)

print(FINAL_PNG)
